# Sentiment Analysis and Topic Modeling of US Presidential Debates from 1960 to 2024

### Daniel Henke (xxx), Sven Kurth (xxx), Margherita Grosso (xxx), Luca Gudi (xxx)

## Install and Load Packages

In [ ]:
!pip install -q gensim 
!pip install -q nltk
!pip install -q spacy
!python -m spacy download en_core_web_sm --quiet
!pip install -q transformers
!pip install -q torch
!pip install -q hf_xet
!pip install -q numpy
%pip uninstall -y gensim numpy > /dev/null 2>&1
%pip install -q numpy==1.26.4
%pip install -q gensim==4.3.1

In [ ]:
import os
import json
import pandas as pd
from collections import defaultdict, Counter
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
from nltk.tokenize import TreebankWordTokenizer
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import hf_xet
import spacy
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim import corpora
from gensim.models.ldaseqmodel import LdaSeqModel

# Load & Prepare Data

In [ ]:
DATA_DIRECTORY = "data_dir"

INCUMBENT_PAIRS = {
    ("ford", "1976"), ("carter", "1980"), ("reagan", "1984"), ("bush", "1984"),
    ("bush", "1992"), ("quayle", "1992"), ("clinton", "1996"), ("gore", "1996"),
    ("bush", "2004"), ("cheney", "2004"), ("obama", "2012"), ("biden", "2012"),
    ("trump", "2020"), ("pence", "2020")
}

WINNER_PAIRS = {
    ("kennedy", "1960"), ("carter", "1976"), ("reagan", "1980"), ("bush", "1980"),
    ("reagan", "1984"), ("bush", "1984"), ("bush", "1988"), ("quayle", "1988"),
    ("clinton", "1992"), ("gore", "1992"), ("clinton", "1996"), ("gore", "1996"),
    ("bush", "2000"), ("cheney", "2000"), ("bush", "2004"), ("cheney", "2004"),
    ("obama", "2008"), ("biden", "2008"), ("obama", "2012"), ("biden", "2012"),
    ("trump", "2016"), ("pence", "2016"), ("biden", "2020"), ("harris", "2020")
}

CANDIDATES = {
    # Presidential
    "kennedy": "Democratic", "nixon": "Republican", "ford": "Republican",
    "carter": "Democratic", "reagan": "Republican", "anderson": "Independent",
    "mondale": "Democratic", "bush": "Republican", "dukakis": "Democratic",
    "clinton": "Democratic", "perot": "Independent", "dole": "Republican",
    "gore": "Democratic", "kerry": "Democratic", "obama": "Democratic",
    "mccain": "Republican", "romney": "Republican", "trump": "Republican",
    "biden": "Democratic",

    # Vice-Presidential
    "ferraro": "Democratic", "quayle": "Republican", "bentsen": "Democratic",
    "kemp": "Republican", "stockdale": "Independent", "lieberman": "Democratic", "cheney": "Republican",
    "edwards": "Democratic", "palin": "Republican", "ryan": "Republican",
    "kaine": "Democratic", "pence": "Republican", "harris": "Democratic",
    "vance": "Republican"
}

VP_CORRECTIONS = ["2000-10-11", "2008-09-26", "2012-10-03"]

In [ ]:
def normalize_last_name(full_name):
    """Extracts the last name and applies capitalization."""
    if not full_name or full_name == "UNKNOWN":
        return "UNKNOWN"
    return full_name.strip().split()[-1].capitalize()

def check_incumbent(last_name, year):
    """Returns True if candidate is incumbent that year."""
    return (last_name.lower(), str(year)) in INCUMBENT_PAIRS

def check_winner(last_name, year):
    """Returns True if candidate won in that year."""
    return (last_name.lower(), str(year)) in WINNER_PAIRS

def check_candidate(last_name):
    """Returns whether person was a candidate and their party if applicable."""
    key = last_name.lower()
    return (key in CANDIDATES), CANDIDATES.get(key)

def get_json_files(directory):
    """Retrieves JSON files excluding partials from a directory."""
    return [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.endswith(".json") and not f.startswith("part")
    ]

def is_vp_debate(content):
    """Determines if a debate is a Vice-Presidential debate."""
    dialogues = [entry.get("dialogue", "").lower() for entry in content[:5]]
    return any("vice presidential" in d for d in dialogues)

def parse_date(date_list):
    """Converts a date list into a datetime.date object or returns 'UNKNOWN'."""
    try:
        return pd.to_datetime(" ".join(date_list)).date()
    except Exception:
        return pd.NaT

def fix_duplicate_names(last_name, year, is_candidate, party):
    """Fixes duplicate names for specific cases."""
    if last_name.lower() == "bush":
        last_name = "Bush Sr" if str(year) in ["1984", "1988", "1992"] else "Bush Jr"
    elif last_name.lower() == "clinton":
        last_name = "Clinton Hillary" if str(year) == "2016" else "Clinton Bill"
    elif last_name.lower() == "edwards" and str(year) == "1960":
        is_candidate = False
        party = None
    return last_name, is_candidate, party

def process_debate_file(file_path):
    """Processes a single JSON debate file and returns a list of parsed rows."""
    rows = []
    with open(file_path, "r", encoding="utf-8") as f:
        debate = json.load(f)
        content = debate.get("content", [])
        date = parse_date(debate.get("date", []))
        year = date.year if pd.notnull(date) else "UNKNOWN"
        vp_flag = is_vp_debate(content)

        for entry in content:
            actor_raw = entry.get("actor", "UNKNOWN")
            dialogue = entry.get("dialogue", "")
            last_name = normalize_last_name(actor_raw)

            is_candidate, party = check_candidate(last_name)
            is_incumbent = check_incumbent(last_name, year)
            is_winner = check_winner(last_name, year)

            # Fix duplicate names and candidate status
            last_name, is_candidate, party = fix_duplicate_names(last_name, year, is_candidate, party)

            rows.append({
                "debate_title" : None,
                "date": date,
                "year": year,
                "actor": last_name,
                "dialogue": dialogue,
                "is_candidate": is_candidate,
                "party": party,
                "is_winner": is_winner,
                "VP_debate": vp_flag,
                "is_incumbent": is_incumbent
            })
    return rows

def correct_vp_debate_flags(df):
    """Manually corrects VP debate flags for known false positives."""
    for d in VP_CORRECTIONS:
        df.loc[df["date"] == pd.to_datetime(d).date(), "VP_debate"] = False
    return df

def generate_debate_titles(df):
    """Add a 'debate_title' column to the DataFrame."""
    debate_titles = {}
    debate_counter = defaultdict(Counter)

    for date, group in df.groupby("date"):
        year = group["year"].iloc[0]
        is_vp = group["VP_debate"].iloc[0]
        
        # Get sorted unique candidate last names
        candidate_names = sorted(set(group[group["is_candidate"]]["actor"]))
        title_base = f"{year}_" + "_".join(candidate_names)

        if is_vp:
            full_title = f"{title_base}_VP"
        else:
            # Number the debate among similar candidate sets in same year
            debate_counter[year][title_base] += 1
            count = debate_counter[year][title_base]
            full_title = f"{title_base}_{count}"

        debate_titles[date] = full_title

    df["debate_title"] = df["date"].map(debate_titles)
    return df

In [ ]:
def debates_to_dataframe(directory):
    """Converts debate JSON files to a DataFrame."""
    
    all_rows = []
    json_files = get_json_files(directory)

    for file_path in json_files:
        all_rows.extend(process_debate_file(file_path))

    df = pd.DataFrame(all_rows)
    df = correct_vp_debate_flags(df)
    df = generate_debate_titles(df)
    return df

df_debates=debates_to_dataframe(DATA_DIRECTORY)
df_debates.head()

In [ ]:
def summarize_debate_actors(
    df,
    only_candidates=True
):
    """
    Summarize actors in a debate DataFrame, counting statements per actor per debate.

    Args:
        df (pd.DataFrame): Debate DataFrame.
        only_candidates (bool): If True, include only candidates.

    Returns:
        pd.DataFrame: Summary with one row per actor per debate.
    """
    # Ensure required columns exist
    required_cols = {"debate_title", "date", "actor", "VP_debate", "is_incumbent", "is_candidate", "party", "dialogue", "is_winner"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"Input DataFrame must contain columns: {required_cols}")

    # Drop duplicates to get one row per actor per debate
    unique_actors = df.drop_duplicates(subset=["date", "actor"])[
        ["debate_title", "date", "actor",  "is_candidate", "party", "is_winner", "VP_debate", "is_incumbent",]
    ]

    # Count number of statements per actor per debate
    statement_counts = df.groupby(["date", "actor"]).size().reset_index(name="statement_count")

    # Merge counts into unique_actors
    unique_actors = unique_actors.merge(statement_counts, on=["date", "actor"], how="left")

    # Optionally filter only candidates
    if only_candidates:
        unique_actors = unique_actors[unique_actors["is_candidate"]]

    # Sort by date and actor
    unique_actors = unique_actors.sort_values(by=["date", "actor"]).reset_index(drop=True)

    return unique_actors


summarize_debate_actors(df_debates)

In [ ]:
def extract_debate_txt(file_path, title, year, date, vp_debate, candidate_info):
    """Extracts structured debate data from a transcript text file.
    Args:
        file_path (str): Path to the transcript text file.
        title (str): Title of the debate.
        year (int): Year of the debate.
        date (str): Date of the debate in 'YYYY-MM-DD' format.
        vp_debate (bool): Whether this is a vice-presidential debate.
        candidate_info (dict): Dictionary mapping speaker last names to:
            {"is_candidate": bool, "party": str, "is_winner": bool, "is_incumbent": bool}
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    date = pd.to_datetime(date, errors="coerce").date() if date else None
    pattern = re.compile(r'^([A-Z][A-Z\s.\-]*)(?:, [A-Z\s.]+)?:\s*(.*)')

    data, current_actor, current_text = [], None, []

    def append_block(actor, text):
        if not actor or not text:
            return
        info = candidate_info.get(actor, {
            'is_candidate': False, 'party': None,
            'is_winner': False, 'is_incumbent': False
        })
        data.append({
            "debate_title": title,  "date": date, "year": year,
            "actor": actor, "dialogue": ' '.join(text).strip(),
            "is_candidate": info['is_candidate'], "party": info['party'],
            "is_winner": info['is_winner'], "VP_debate": vp_debate,
            "is_incumbent": info['is_incumbent']
        })

    for line in lines:
        match = pattern.match(line)
        if match:
            append_block(current_actor, current_text)
            current_actor = match.group(1).split()[-1].title()
            current_text = [match.group(2)] if match.group(2) else []
        else:
            current_text.append(line)

    append_block(current_actor, current_text)
    return pd.DataFrame(data)

In [ ]:
debate_1992_first_half=extract_debate_txt(
    file_path="data_dir/transcript_1992_oct_15_first_half.txt",
    title="1992_Bush Sr_Clinton Bill_Perot_2",
    year=1992, date="1992-10-15", vp_debate=False,
    candidate_info={
        "Bush": {"is_candidate": True, "party": "Republican","is_winner": False,"is_incumbent": True},
        "Clinton": {"is_candidate": True,"party": "Democratic","is_winner": True,"is_incumbent": False},
        "Perot": {"is_candidate": True,"party": "Independent","is_winner": False,"is_incumbent": False}
    }
)

#Rename Bush to Bush Sr and Clinton to Clinton (Bill)
debate_1992_first_half.loc[debate_1992_first_half["actor"] == "Bush", "actor"] = "Bush Sr"
debate_1992_first_half.loc[debate_1992_first_half["actor"] == "Clinton", "actor"] = "Clinton Bill"
debate_1992_first_half.head(20)

In [ ]:
#concat with the rest of the data
df_debates = pd.concat([ debate_1992_first_half, df_debates], ignore_index=True)
summary=summarize_debate_actors(df_debates)
summary[summary["date"]==pd.to_datetime("1992-10-15").date()]

In [ ]:
debate_2024_biden=extract_debate_txt(
    file_path="data_dir/transcript_2024_Trump_Biden.txt",
    title="2024_Trump_Biden",
    year=2024,date="2024-07-27",vp_debate=False,
    candidate_info={
        "Trump": {"is_candidate": True,"party": "Republican","is_winner": True,"is_incumbent": False},
        "Biden": {"is_candidate": True,"party": "Democratic","is_winner": False,"is_incumbent": True}
    }
)

df_debates = pd.concat([df_debates, debate_2024_biden], ignore_index=True)
debate_2024_biden.head(20)

In [ ]:
debate_2024_harris=extract_debate_txt(
    file_path="data_dir/transcript_2024_Trump_Harris.txt",
    title="2024_Trump_Harris",
    year=2024,date="2024-09-10",vp_debate=False,
    candidate_info={
        "Trump": {"is_candidate": True,"party": "Republican","is_winner": True,"is_incumbent": False},
        "Harris": {"is_candidate": True,"party": "Democratic","is_winner": False,"is_incumbent": False}
    }
)

df_debates = pd.concat([df_debates, debate_2024_harris], ignore_index=True)
debate_2024_harris.head(20)

In [ ]:
debate_2024_vp=extract_debate_txt(
    file_path="data_dir/transcript_2024_Vance_Walz.txt",
    title="2024_Vance_Walz_VP",
    year=2024,date="2024-10-01",vp_debate=True,
    candidate_info={
        "Jdv": {"is_candidate": True,"party": "Republican","is_winner": True,"is_incumbent": False},
        "Tw": {"is_candidate": True,"party": "Democratic","is_winner": False,"is_incumbent": False}
    }
)

#Rename Jdv to Vance, Tw to Walz, No to O'Donnell, and Mb to Brennan
debate_2024_vp.loc[debate_2024_vp["actor"] == "Jdv", "actor"] = "Vance"
debate_2024_vp.loc[debate_2024_vp["actor"] == "Tw", "actor"] = "Walz"
debate_2024_vp.loc[debate_2024_vp["actor"] == "No", "actor"] = "O'Donnell"
debate_2024_vp.loc[debate_2024_vp["actor"] == "Mb", "actor"] = "Brennan"


df_debates = pd.concat([df_debates, debate_2024_vp], ignore_index=True)
debate_2024_vp.head(20)

In [ ]:
df_debates.sort_values(by=["year", "date"], inplace=True)
df_debates.reset_index(drop=True, inplace=True)
df_debates

In [ ]:
# Save the final DataFrame to a CSV file
df_debates.to_csv("debate_transcripts_cleaned.csv", index=False, encoding="utf-8")
print(f"Data saved")

# Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_debate['word_count'] = df_debate['dialogue'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 6))
sns.histplot(df_debate['word_count'], bins=50, color='#4C72B0', kde=True)
plt.title('Distribution of Sentence Lengths (in Words)')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
# Keep only candidates and filter outliers above 600 words
df_candidates = df_debate[df_debate['is_candidate'] == True].copy()
df_candidates['word_count'] = df_candidates['dialogue'].apply(lambda x: len(str(x).split()))

# Create 3-year election blocks
def map_year_block(y):
    if y < 2000:
        return "1960–1996"
    elif y <= 2012:
        return "2000–2012"
    else:
        return "2016–2024"

df_candidates['year_block'] = df_candidates['year'].apply(map_year_block)
# Plot
g = sns.displot(
    data=df_candidates,
    x='word_count',
    col='year_block',
    col_order=["1960–1996", "2000–2012", "2016–2024"],
    col_wrap=3,
    height=4,
    aspect=1.4,
    bins=40,
    kde=True,
    color='#4C72B0',
    facet_kws={'sharey': True}
)

g.set_titles("Period: {col_name}")
g.set_axis_labels("Words per Sentence", "Frequency")
plt.subplots_adjust(top=0.85)
g.fig.suptitle("Sentence Length Distribution", fontsize=16)
plt.show()

In [ ]:
# Make sure word_count exists
df_debate['word_count'] = df_debate['dialogue'].apply(lambda x: len(str(x).split()))

# Overall average sentence length
average_sentence_length = df_debate['word_count'].mean()
print(f"Average sentence length: {average_sentence_length:.2f} words")

In [ ]:
# Ensure word count column exists
df_debate['word_count'] = df_debate['dialogue'].apply(lambda x: len(str(x).split()))

# Filter only candidate speech
df_candidates = df_debate[df_debate['is_candidate'] == True]

# Calculate average sentence length
average_sentence_length = df_candidates['word_count'].mean()
print(f"Average sentence length (candidates only): {average_sentence_length:.2f} words")


In [ ]:
# Define your custom eras
def assign_time_span(year):
    if year <= 1980:
        return '1960–1980'
    elif year <= 2004:
        return '1984–2004'
    else:
        return '2008–2024'

df_candidates['time_span'] = df_candidates['year'].apply(assign_time_span)

# Compute average per span
avg_per_span = df_candidates.groupby('time_span')['word_count'].mean().reset_index(name='avg_sentence_length')
print(avg_per_span)

In [ ]:
words_per_candidate = df_debate[(df_debate['is_candidate'] == True) & (df_debate['VP_debate'] == False)].groupby('actor')['word_count'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
words_per_candidate.plot(kind='bar', color='#4C72B0')
plt.title('Total Words Spoken by Each Candidate')
plt.ylabel('Total Word Count')
plt.xlabel('Candidate')
plt.xticks(rotation=45, size=10)
plt.yticks(size=8.5)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Step 1: Filter presidential candidate data
df_debate['word_count'] = df_debate['dialogue'].apply(lambda x: len(str(x).split()))
pres_debate = df_debate[(df_debate['is_candidate'] == True) & (df_debate['VP_debate'] == False)]

# Step 2: Words per candidate per debate
words_per_debate = pres_debate.groupby(['actor', 'date'])['word_count'].sum().reset_index()
avg_words_per_candidate = words_per_debate.groupby('actor')['word_count'].mean().sort_values(ascending=False)

# Step 3: Map each candidate to their party (take the most frequent party per actor)
actor_party_map = pres_debate.groupby('actor')['party'].agg(lambda x: x.mode().iloc[0])

# Step 4: Get list of colors by actor
bar_colors = [party_palette.get(actor_party_map.get(actor, 'Independent'), '#808080') for actor in avg_words_per_candidate.index]

# Step 5: Plot
plt.figure(figsize=(8, 5))
avg_words_per_candidate.plot(
    kind='bar',
    color=bar_colors
)

legend_elements = [
    Patch(facecolor='#007FFF', label='Democratic'),
    Patch(facecolor='#d62728', label='Republican'),
    Patch(facecolor='#FFBF00', label='Independent')
]
plt.legend(handles=legend_elements, title='Party', loc='upper right')

plt.title('Average Words per Presidential Debate by Candidate', size=14)
plt.ylabel('Avg. Word Count per Debate')
plt.xlabel('Candidate')
plt.xticks(rotation=60, size=9)
plt.yticks(size=9)
plt.tight_layout()
plt.show()

In [ ]:
# Filter candidates only
df_candidates = df_debate[df_debate['is_candidate'] == True].copy()
df_candidates['word_count'] = df_candidates['dialogue'].apply(lambda x: len(str(x).split()))
#df_candidates = df_candidates[df_candidates['word_count'] <= 600] 

# Custom party color palette 
party_palette = {
    'Democratic': '#007FFF',   
    'Republican': '#d62728',    
    'Independent': '#FFBF00'   
}

plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df_candidates,
    x='party',
    y='word_count',
    palette=party_palette,
    showfliers=True,
    width=0.6
)

plt.title('Sentence Length by Party', fontsize=16)
plt.xlabel('Party', fontsize=12)
plt.ylabel('Word Count', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()